[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/11_agents_tools_mcp/41_model_context_protocol.ipynb)

# 📓 Notebook 41 — The Model Context Protocol (MCP)

> **Module:** Agents, Tools & MCP · **Estimated time:** 80–100 min · **Difficulty:** Advanced

You now know how to build tools (NB 40) and wire them into an agent (NB 39). But every framework wired them *differently* — an OpenAI tool, a LangChain tool and a Claude tool are described in three incompatible ways. **The Model Context Protocol (MCP)** fixes that. Open-sourced by **Anthropic** in late 2024 and now supported across the ecosystem, MCP is a single open standard for how an AI application connects to tools, data, and prompts — often called *"a USB-C port for AI"*.

Write one MCP **server** and it works in **Claude Desktop, Claude Code, and any other MCP host** — no per-app glue.

This notebook builds a **working MCP server and client from scratch, in process**, speaking the real protocol (JSON-RPC 2.0, the real method names and message shapes) so you understand exactly what the SDK does for you. It runs **100% offline**. The real `mcp` Python SDK and the Claude host config are shown as drop-in references.

## 🎯 Learning objectives

By the end you can:

1. Explain MCP's **host / client / server** architecture and when to use it.
2. Name the three server **primitives** — **tools**, **resources**, **prompts** — and who controls each.
3. Trace the **JSON-RPC** lifecycle: `initialize` → `tools/list` → `tools/call` → …
4. Implement a minimal MCP **server** and **client** and exchange messages over a transport.
5. Connect an **agent** to a server so it discovers and calls tools it never hard-coded.
6. Map the offline build onto the real **`mcp` SDK** and a **Claude Desktop / Claude Code** config.

## ✅ Prerequisites

Notebooks 39–40 (agents and tools), JSON (NB 4). A real LLM is *not* required.

## 1. Why MCP exists

Before MCP, every integration was **M×N**: M apps each needing custom code for N tools/data sources. MCP makes it **M+N**: each app speaks MCP once, each tool exposes MCP once, and they all interoperate.

```
        Without MCP                         With MCP
   app1 ── glue ── toolA              app1 ─┐        ┌─ serverA (toolA)
   app1 ── glue ── toolB                    ├─ MCP ──┤
   app2 ── glue ── toolA              app2 ─┘        └─ serverB (toolB)
   app2 ── glue ── toolB
   (every line is bespoke)           (one standard on each side)
```

**Architecture — three roles:**

| Role | What it is | Examples |
|---|---|---|
| **Host** | the AI application the user interacts with | Claude Desktop, Claude Code, an IDE, your app |
| **Client** | lives inside the host; keeps a 1:1 connection to one server | the host's MCP connector |
| **Server** | exposes capabilities over MCP | a filesystem server, a GitHub server, *your* server |

The model lives in the host. The server never sees your API keys; the host mediates everything.

## 2. The three primitives — and who controls them

A server can expose three kinds of capability. The distinction is **who decides to use it**:

| Primitive | Controlled by | Analogy | Example |
|---|---|---|---|
| **Tools** | the **model** | a POST endpoint | `create_ticket(...)`, `run_query(...)` |
| **Resources** | the **application** | a GET endpoint / a file | `data://orders`, `file:///readme.md` |
| **Prompts** | the **user** | a slash-command template | `/triage`, `/summarise-thread` |

We'll implement all three. First, the wire format they all travel on.

## 3. The wire protocol: JSON-RPC 2.0

MCP messages are **JSON-RPC 2.0**. A request has an `id`, a `method`, and `params`; a response echoes the `id` and carries either a `result` or an `error`. That's the whole envelope.

In [ ]:
import json

def rpc_request(id, method, params=None):
    return {"jsonrpc": "2.0", "id": id, "method": method, "params": params or {}}

def rpc_result(id, result):
    return {"jsonrpc": "2.0", "id": id, "result": result}

def rpc_error(id, code, message):
    return {"jsonrpc": "2.0", "id": id, "error": {"code": code, "message": message}}

print(json.dumps(rpc_request(1, "tools/list"), indent=1))
print(json.dumps(rpc_result(1, {"tools": ["..."]}), indent=1))

## 4. A minimal MCP server

The server registers **tools**, **resources**, and **prompts**, then routes incoming JSON-RPC requests by `method`. The method names and result shapes below are the *real* MCP ones (`tools/list`, `tools/call`, `resources/read`, `prompts/get`, …) — when you switch to the SDK, only the transport changes.

In [ ]:
class MethodNotFound(Exception): pass   # JSON-RPC -32601
class InvalidParams(Exception): pass    # JSON-RPC -32602

class MCPServer:
    def __init__(self, name, version="0.1.0"):
        self.name, self.version = name, version
        self.tools, self.resources, self.prompts = {}, {}, {}

    # --- registration -------------------------------------------------------
    def add_tool(self, name, description, input_schema, fn):
        self.tools[name] = {"description": description, "inputSchema": input_schema, "fn": fn}

    def add_resource(self, uri, name, fn, mime="application/json"):
        self.resources[uri] = {"name": name, "mimeType": mime, "fn": fn}

    def add_prompt(self, name, description, fn, arguments=None):
        self.prompts[name] = {"description": description, "arguments": arguments or [], "fn": fn}

    # --- JSON-RPC dispatch --------------------------------------------------
    def handle(self, req: dict) -> dict:
        rid, method, params = req.get("id"), req["method"], req.get("params", {})
        try:
            return rpc_result(rid, self._route(method, params))
        except MethodNotFound as e:
            return rpc_error(rid, -32601, f"method not found: {e}")   # unknown method
        except InvalidParams as e:
            return rpc_error(rid, -32602, f"invalid params: {e}")     # unknown tool/uri/prompt name
        except Exception as e:                                        # noqa: BLE001
            return rpc_error(rid, -32603, f"internal error: {e}")     # tool raised

    def _route(self, method, params):
        if method == "initialize":
            return {"protocolVersion": "2025-06-18",
                    "capabilities": {"tools": {}, "resources": {}, "prompts": {}},
                    "serverInfo": {"name": self.name, "version": self.version}}
        if method == "tools/list":
            return {"tools": [{"name": n, "description": t["description"],
                               "inputSchema": t["inputSchema"]} for n, t in self.tools.items()]}
        if method == "tools/call":
            if params["name"] not in self.tools:
                raise InvalidParams(f"unknown tool '{params['name']}'")
            t = self.tools[params["name"]]
            result = t["fn"](**params.get("arguments", {}))
            return {"content": [{"type": "text", "text": json.dumps(result)}], "isError": False}
        if method == "resources/list":
            return {"resources": [{"uri": u, "name": r["name"], "mimeType": r["mimeType"]}
                                  for u, r in self.resources.items()]}
        if method == "resources/read":
            if params["uri"] not in self.resources:
                raise InvalidParams(f"unknown resource '{params['uri']}'")
            r = self.resources[params["uri"]]
            return {"contents": [{"uri": params["uri"], "mimeType": r["mimeType"],
                                  "text": json.dumps(r["fn"]())}]}
        if method == "prompts/list":
            return {"prompts": [{"name": n, "description": p["description"],
                                 "arguments": p["arguments"]} for n, p in self.prompts.items()]}
        if method == "prompts/get":
            if params["name"] not in self.prompts:
                raise InvalidParams(f"unknown prompt '{params['name']}'")
            p = self.prompts[params["name"]]
            return {"messages": p["fn"](**params.get("arguments", {}))}
        raise MethodNotFound(method)

print("MCPServer defined")

## 5. Populate a server: the "support-ops" server

We expose two **tools**, one **resource**, and one **prompt** over a tiny support dataset.

In [ ]:
CSAT = {"chat": 4.1, "email": 3.6, "phone": 4.4, "social": 3.2}
TICKETS = [{"id": 1, "channel": "email", "status": "open"},
           {"id": 2, "channel": "chat",  "status": "closed"},
           {"id": 3, "channel": "email", "status": "open"}]

server = MCPServer("support-ops")

# Tools (model-controlled)
server.add_tool(
    "get_csat", "Average customer satisfaction (1-5) for a support channel.",
    {"type": "object", "properties": {"channel": {"type": "string"}}, "required": ["channel"]},
    lambda channel: {"channel": channel, "csat": CSAT.get(channel.lower(), None)})

server.add_tool(
    "count_tickets", "Count tickets with a given status (open/closed).",
    {"type": "object", "properties": {"status": {"type": "string", "enum": ["open", "closed"]}},
     "required": ["status"]},
    lambda status: {"status": status, "count": sum(t["status"] == status for t in TICKETS)})

# Resource (application-controlled) — addressed by URI
server.add_resource("data://tickets", "All tickets", lambda: TICKETS)

# Prompt (user-controlled) — a reusable template
server.add_prompt(
    "triage", "Draft a triage note for a channel.",
    lambda channel: [{"role": "user", "content": {"type": "text",
        "text": f"Summarise open issues for the {channel} channel and propose next steps."}}],
    arguments=[{"name": "channel", "required": True}])

print("server primitives -> tools:", list(server.tools),
      "| resources:", list(server.resources), "| prompts:", list(server.prompts))

## 6. The transport + client

A **transport** carries JSON-RPC bytes between client and server. Real MCP uses **stdio** (a local subprocess) or **Streamable HTTP** (remote). Offline we use an in-process transport that hands the dict straight to the server's `handle()` — same messages, no sockets.

In [ ]:
class InProcessTransport:
    """Stand-in for stdio/HTTP: a request goes in, a response comes back."""
    def __init__(self, server): self.server = server
    def round_trip(self, request: dict) -> dict: return self.server.handle(request)

class MCPClient:
    def __init__(self, transport): self.t = transport; self._id = 0; self.server_info = None

    def _call(self, method, params=None):
        self._id += 1
        resp = self.t.round_trip(rpc_request(self._id, method, params))
        if "error" in resp:
            raise RuntimeError(f"{method} failed: {resp['error']['message']}")
        return resp["result"]

    def initialize(self):
        self.server_info = self._call("initialize")["serverInfo"]
        return self.server_info
    def list_tools(self):        return self._call("tools/list")["tools"]
    def call_tool(self, name, **arguments):
        out = self._call("tools/call", {"name": name, "arguments": arguments})
        return json.loads(out["content"][0]["text"])      # unwrap the text content
    def list_resources(self):    return self._call("resources/list")["resources"]
    def read_resource(self, uri):
        return json.loads(self._call("resources/read", {"uri": uri})["contents"][0]["text"])
    def list_prompts(self):      return self._call("prompts/list")["prompts"]
    def get_prompt(self, name, **arguments):
        return self._call("prompts/get", {"name": name, "arguments": arguments})["messages"]

client = MCPClient(InProcessTransport(server))
print("connected to:", client.initialize())

## 7. The full lifecycle — discover, then use

A host always **initializes**, then **lists** what a server offers, then **calls**. Watch each JSON-RPC round-trip:

In [ ]:
print("🧰 TOOLS")
for t in client.list_tools():
    print(f"   {t['name']}: {t['description']}")
print("   get_csat('phone') ->", client.call_tool("get_csat", channel="phone"))
print("   count_tickets('open') ->", client.call_tool("count_tickets", status="open"))

print("\n📄 RESOURCES")
for r in client.list_resources():
    print(f"   {r['uri']} ({r['mimeType']}) — {r['name']}")
print("   read data://tickets ->", client.read_resource("data://tickets"))

print("\n📝 PROMPTS")
for p in client.list_prompts():
    print(f"   /{p['name']}: {p['description']}")
print("   get triage(channel='email') ->")
print("   ", client.get_prompt("triage", channel="email")[0]["content"]["text"])

## 8. Connect an agent to the MCP server

Here's the payoff. An agent doesn't hard-code tools any more — it **discovers** them from the server at runtime via `list_tools()`, and executes them via `call_tool()`. The same agent works against *any* MCP server.

In [ ]:
import re

def mcp_agent(question: str, client: MCPClient, verbose=True):
    """Toy router (stand-in for the model) over whatever tools the server advertises."""
    tools = {t["name"]: t for t in client.list_tools()}      # discovered, not hard-coded
    q = question.lower()
    # pick a tool by matching its description words to the question
    pick, best = None, 0
    for name, t in tools.items():
        score = len(set(t["description"].lower().split()) & set(q.split()))
        if score > best: pick, best = name, score
    if pick is None:
        return "No suitable tool advertised."
    # extract an argument crudely (a real model fills the inputSchema)
    if pick == "get_csat":
        ch = next((c for c in CSAT if c in q), "chat"); args = {"channel": ch}
    else:
        st = "open" if "open" in q else "closed"; args = {"status": st}
    if verbose: print(f"💭 chose {pick}({args}) from {list(tools)}")
    result = client.call_tool(pick, **args)
    return result

print(mcp_agent("what is the csat for phone?", client))
print(mcp_agent("how many open tickets are there?", client))

## 9. The real thing — `mcp` SDK & Claude hosts

Offline we wrote the plumbing by hand. In production you use the official **`mcp`** Python SDK (`pip install "mcp[cli]"`), whose **FastMCP** turns plain functions into a server with decorators — the protocol is identical to what you just built.

In [ ]:
# ── Real MCP server (reference) — pip install "mcp[cli]" ─────────────────────
# from mcp.server.fastmcp import FastMCP
#
# mcp = FastMCP("support-ops")
#
# @mcp.tool()
# def get_csat(channel: str) -> dict:
#     """Average customer satisfaction (1-5) for a support channel."""
#     return {"channel": channel, "csat": CSAT.get(channel.lower())}
#
# @mcp.resource("data://tickets")
# def tickets() -> list: return TICKETS
#
# @mcp.prompt()
# def triage(channel: str) -> str:
#     return f"Summarise open issues for the {channel} channel and propose next steps."
#
# if __name__ == "__main__":
#     mcp.run(transport="stdio")          # or transport="streamable-http" for remote
print("(reference — the in-process server above is the runnable version)")

**Registering your server with a Claude host:**

*Claude Desktop* — add to `claude_desktop_config.json`:

```json
{
  "mcpServers": {
    "support-ops": {
      "command": "python",
      "args": ["/abs/path/to/support_ops_server.py"]
    }
  }
}
```

*Claude Code* — one command:

```bash
claude mcp add support-ops -- python /abs/path/to/support_ops_server.py
```

Once registered, the host calls `tools/list` on startup and the model can use your tools in any conversation — exactly the lifecycle from §7, over stdio instead of in-process.

In [ ]:
# ── Real MCP client (reference) — connect to a server over stdio ────────────
# from mcp import ClientSession, StdioServerParameters
# from mcp.client.stdio import stdio_client
#
# params = StdioServerParameters(command="python", args=["support_ops_server.py"])
# async def main():
#     async with stdio_client(params) as (read, write):
#         async with ClientSession(read, write) as session:
#             await session.initialize()
#             tools = await session.list_tools()
#             result = await session.call_tool("get_csat", {"channel": "phone"})
#             print(result)
# # asyncio.run(main())
print("(reference — see MCPClient above for the runnable, sync version)")

## 🧪 Practice exercises

### Exercise 1 — ⭐ Add a tool to the live server

Register a `list_channels` tool, then call it through the client (no client changes needed — that's the point of MCP).

In [ ]:
server.add_tool("list_channels", "List all known support channels.",
                {"type": "object", "properties": {}},
                lambda: {"channels": list(CSAT)})
print(client.call_tool("list_channels"))

### Exercise 2 — ⭐⭐ Add a second resource

Expose `data://csat` returning the CSAT dict, then read it through the client.

In [ ]:
server.add_resource("data://csat", "CSAT by channel", lambda: CSAT)
print([r["uri"] for r in client.list_resources()])
print(client.read_resource("data://csat"))

### Exercise 3 — ⭐⭐ Inspect a raw round-trip

Send a `tools/list` request *directly* through the transport and print the raw JSON-RPC response envelope (what really crosses the wire).

In [ ]:
raw = InProcessTransport(server).round_trip(rpc_request(99, "tools/list"))
print(json.dumps(raw, indent=1)[:400], "...")

### Exercise 4 — ⭐⭐ Debug me 🐞

This call passes the wrong argument name, so the server can't run the tool and returns a JSON-RPC error (surfaced here as a `RuntimeError`). Read it, then fix the call (next cell).

In [ ]:
# 🐞 BUG (INTENTIONALLY ERRORS): the tool's inputSchema wants 'channel', not 'chanel'.
print(client.call_tool("get_csat", chanel="chat"))

In [ ]:
# ✅ Fix: use the argument names from the tool's inputSchema.
print(client.call_tool("get_csat", channel="chat"))

## 🧠 Stretch exercises

### Stretch A — ⭐⭐⭐ Error on unknown method

Send a request with a bogus method (`tools/destroy`) and confirm the server returns a JSON-RPC `error` with code `-32601`, not a crash.

In [ ]:
resp = InProcessTransport(server).round_trip(rpc_request(7, "tools/destroy"))
print(resp)
assert "error" in resp and resp["error"]["code"] == -32601

### Stretch B — ⭐⭐⭐ A logging transport

Wrap the transport so it prints every request/response — the MCP equivalent of `tcpdump`, invaluable when a server misbehaves.

In [ ]:
class LoggingTransport(InProcessTransport):
    def round_trip(self, request):
        print("→", request["method"], request.get("params", {}))
        resp = super().round_trip(request)
        print("←", "error" if "error" in resp else "ok")
        return resp

dbg = MCPClient(LoggingTransport(server)); dbg.initialize()
dbg.call_tool("count_tickets", status="open")

### Stretch C — ⭐⭐⭐ Two servers, one client-side router

Real hosts connect to *several* servers at once. Build a `MultiServer` that initializes two servers and routes a tool call to whichever advertises that tool.

In [ ]:
math_server = MCPServer("math")
math_server.add_tool("add", "Add two numbers.",
    {"type": "object", "properties": {"a": {"type": "number"}, "b": {"type": "number"}},
     "required": ["a", "b"]}, lambda a, b: {"sum": a + b})

class MultiServer:
    def __init__(self, servers):
        self.clients = [MCPClient(InProcessTransport(s)) for s in servers]
        for c in self.clients: c.initialize()
    def call(self, tool, **args):
        for c in self.clients:
            if tool in {t["name"] for t in c.list_tools()}:
                return c.call_tool(tool, **args)
        raise KeyError(f"no server offers '{tool}'")

multi = MultiServer([server, math_server])
print(multi.call("get_csat", channel="email"))   # support-ops
print(multi.call("add", a=2, b=3))                # math

### Stretch D — ⭐⭐⭐ Resource templates

Real MCP supports URI *templates* like `ticket://{id}`. Add a resolver so `read_resource("ticket://2")` returns ticket #2.

In [ ]:
def read_templated(uri: str):
    if uri.startswith("ticket://"):
        tid = int(uri.split("://")[1])
        return next((t for t in TICKETS if t["id"] == tid), {"error": "not found"})
    raise KeyError(uri)

print(read_templated("ticket://2"))
print(read_templated("ticket://99"))

## 🎁 Bonus mini-project — a capability report

Write `describe(client)` that initializes a connection and prints a tidy catalogue of everything a server offers — tools, resources, and prompts — the "what can this server do?" card a host might show a user.

In [ ]:
def describe(client: MCPClient) -> dict:
    info = client.initialize()
    cat = {"server": info,
           "tools": [t["name"] for t in client.list_tools()],
           "resources": [r["uri"] for r in client.list_resources()],
           "prompts": [p["name"] for p in client.list_prompts()]}
    print(json.dumps(cat, indent=2))
    return cat

_ = describe(MCPClient(InProcessTransport(server)))

## 🧠 Key takeaways

1. **MCP is one open standard** (from Anthropic) for connecting AI apps to tools, data, and prompts — turning M×N glue into M+N.
2. The roles are **host** (the app + model), **client** (1:1 connector), and **server** (your capabilities).
3. Servers expose three primitives: **tools** (model-controlled), **resources** (app-controlled), **prompts** (user-controlled).
4. The wire format is **JSON-RPC 2.0**; the lifecycle is `initialize` → `*/list` → `*/call|read|get`.
5. **Transport is pluggable** — stdio locally, Streamable HTTP remotely; the messages don't change.
6. An MCP agent **discovers** tools at runtime, so one server works in Claude Desktop, Claude Code, and your own app unchanged.
7. The **`mcp` SDK** (FastMCP) writes all this plumbing for you — you just decorate functions.

## ✅ Self-assessment

- [ ] Explain host vs client vs server, and where the model and the keys live
- [ ] State which of tools/resources/prompts the model, app, and user each control
- [ ] Trace a JSON-RPC `initialize` → `tools/list` → `tools/call` exchange
- [ ] Implement a server that routes those methods and a client that calls them
- [ ] Connect an agent that discovers tools at runtime
- [ ] Sketch the equivalent FastMCP server and a Claude host config

## 🚀 Next step

Continue with **Notebook 42 — Multi-Agent Systems**, where several specialist agents — each backed by tools and MCP servers — collaborate under an orchestrator to solve a task no single agent should own.